In [16]:
import os
import json
import pandas as pd

from modules.config import *

In [8]:
model = 'all_correct_win4_b16_kbrs_20250915_072116/model_030'
replay = '36.rep'
label_method = 'all_correct'

prediction_path = os.path.join(PREDICTIONS_DIR, model, replay, f'{label_method}.json')

prediction_path

'/workspace/predictions/all_correct_win4_b16_kbrs_20250915_072116/model_030/36.rep/all_correct.json'

In [11]:
with open(prediction_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [19]:
annotation = pd.DataFrame(data['annotations'])

In [21]:
annotation.head(5)

,id,image_id,category_id,bbox,score,area,segmentation,iscrowd
0,1,3,1,"[0, 1, 19, 12]",0.962068,228,"[[0, 1, 19, 1, 19, 13, 0, 13]]",0
1,2,3,1,"[108, 112, 19, 11]",0.899726,209,"[[108, 112, 127, 112, 127, 123, 108, 123]]",0
2,3,3,1,"[5, 3, 20, 11]",0.307807,220,"[[5, 3, 25, 3, 25, 14, 5, 14]]",0
3,4,3,1,"[107, 115, 20, 12]",0.232050,240,"[[107, 115, 127, 115, 127, 127, 107, 127]]",0
4,5,3,1,"[108, 2, 19, 12]",0.217142,228,"[[108, 2, 127, 2, 127, 14, 108, 14]]",0


In [24]:
def dataframe_topk_score(dataframe, K):
    topk_df = (
      dataframe.sort_values('score', ascending=False)
        .groupby(['image_id'], group_keys=False)
        .head(K)
    )
    return topk_df.sort_values(['image_id'])

In [25]:
res = dataframe_topk_score(annotation, K=1)

In [31]:
res.head(5)

,id,image_id,category_id,bbox,score,area,segmentation,iscrowd
0,1,3,1,"[0, 1, 19, 12]",0.962068,228,"[[0, 1, 19, 1, 19, 13, 0, 13]]",0
8,9,4,1,"[0, 1, 19, 12]",0.962068,228,"[[0, 1, 19, 1, 19, 13, 0, 13]]",0
16,17,5,1,"[0, 1, 19, 12]",0.962068,228,"[[0, 1, 19, 1, 19, 13, 0, 13]]",0
24,25,6,1,"[0, 1, 19, 12]",0.962068,228,"[[0, 1, 19, 1, 19, 13, 0, 13]]",0
32,33,7,1,"[0, 1, 19, 12]",0.962068,228,"[[0, 1, 19, 1, 19, 13, 0, 13]]",0


In [33]:
vpx, vpy = [], []
for bbox in res['bbox']:
    x, y, w, h = map(int, bbox)
    vpx.append(x)
    vpy.append(y)

In [35]:
res['vpx'] = vpx
res['vpy'] = vpy

In [38]:
vpd = res[['image_id', 'vpx', 'vpy']]

In [47]:
vpd['vpx'] = vpd['vpx'] * 32
vpd['vpy'] = vpd['vpy'] * 32

/tmp/ipykernel_132/3891000309.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vpd['vpx'] = vpd['vpx'] * 32
/tmp/ipykernel_132/3891000309.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vpd['vpy'] = vpd['vpy'] * 32


In [48]:
vpd.columns = ['frame', 'vpx', 'vpy']

In [49]:
vpd.to_csv('36.rep.vpd', index=False)

In [52]:
vpd.loc[vpd['vpx'] == 0].loc[vpd['vpy'] == 32]

,frame,vpx,vpy
0,3,0,32
8,4,0,32
16,5,0,32
24,6,0,32
32,7,0,32
...,...,...,...
949133,23601,0,32
949186,23602,0,32
949243,23603,0,32
949300,23604,0,32
